Get SMOLSent En->Cantonese data

In [20]:
import datasets

en_yue_sentences: datasets.dataset_dict.DatasetDict = datasets.load_dataset("google/smol", "smolsent__en_yue") # type: ignore

Generating train split: 863 examples [00:00, 59849.28 examples/s]


Benchmark sentences on various translation sites using bleu to demonstrate that chinese models don't necessarily outperform cantonese models

In [189]:
import translators as ts
import time

services = ["bing", "google"]

langs = {
    "bing": ["chinese (simplified)", "chinese (traditional)", "cantonese"],
    "google": ["chinese"]
}

lang_map = {
    "bing": {"chinese (simplified)" : "zh-Hans", "chinese (traditional)": "zh-Hant", "cantonese": "yue", "english": "en"},
    "google": {"chinese": "zh", "english": "en"}
}

from tqdm import tqdm

all_trans = []

sentence_subset = en_yue_sentences["train"].select(range(10))

for sentence in tqdm(sentence_subset):
    assert type(sentence) == dict
    
    translations = {
        "original_eng": sentence["src"],
        "original_yue": sentence["trg"]
    }

    for service in services:

        attempts = 0
        while attempts < 3:
            try:
                trans = {language: {"translation":ts.translate_text(sentence["trg"], translator="bing", from_language=lang_map[service][language], to_language=lang_map[service]["english"])} for language in langs[service] }
                translations[service] = trans
                break
            except:
                tqdm.write(f"error on sentece {sentence['id']}, retrying... ({attempts + 1})")
                attempts += 1
                time.sleep(3)
        if attempts == 3:
            tqdm.write(f"skipped sentence {sentence['id']} after {attempts+1} attempts")
            continue
        
    all_trans.append(translations)

100%|██████████| 10/10 [00:30<00:00,  3.06s/it]


In [159]:
from nltk.translate import bleu_score

import numpy as np

def bleu(ref, cand):
    return bleu_score.sentence_bleu(
            [ref.split()], 
            cand.split(), 
                #use smoothing method 7 that had best chinese->english human-evaluation corerlation from https://aclanthology.org/W14-3346/
            smoothing_function= bleu_score.SmoothingFunction().method7 
            )

def benchmark(all_trans, metric, metric_name): 
    with tqdm(total = len(all_trans) * np.sum([len(langs[service]) for service in services]), leave=True) as pbar:
        for i, trans in enumerate(all_trans):
            for service in services:
                for lang in langs[service]:
                    all_trans[i][service][lang][metric_name] = metric(trans["original_eng"], trans[service][lang]["translation"])
                    pbar.update()

def print_benchmark(all_trans, metric_name):
    for service in services:
            for lang in langs[service]:
                print(f'{service:10s} - {lang:25s}: {np.mean([trans[service][lang][metric_name] for trans in all_trans]):.4f} BLEU')

In [160]:
benchmark(all_trans, bleu, "bleu")
print_benchmark(all_trans, "bleu")

100%|██████████| 40/40 [00:00<00:00, 2858.37it/s]

bing       - chinese (simplified)     : 0.2547 BLEU
bing       - chinese (traditional)    : 0.2074 BLEU
bing       - cantonese                : 0.2538 BLEU
google     - chinese                  : 0.2632 BLEU


Since BLEU is a n-gram based evaluation proxy it evaluates sentences badly when they don't share vocabulary or word order with the reference, even if the semantic meaning is conserved. This can be seen below:  

In [183]:
def print_sentences(sent_trans):
    print(f'{"original":29s} : {sent_trans["original_eng"]}')
    [print(f'bing   {lang:22s} : {sent_trans["bing"][lang]["translation"]}') for lang in langs["bing"]]
    [print(f'google {lang:22s} : {sent_trans["google"][lang]["translation"]}') for lang in langs["google"]]
    print(f'{"original yue":29s} : {sent_trans["original_yue"]}')

In [184]:
print_sentences(all_trans[4])
print("\n")
print_benchmark([all_trans[4]], "bleu")

original                      : But faster economic activity could also translate into some degree of inflation.
bing   chinese (simplified)   : However, accelerated economic activity may also bring a certain degree of inflation.
bing   chinese (traditional)  : However, accelerated economic activity may also bring a certain degree of inflation.
bing   cantonese              : However, accelerated economic activity could also bring about a certain degree of inflation.
google chinese                : However, accelerated economic activity may also bring a certain degree of inflation.
original yue                  : 但加速嘅經濟活動都有機會帶嚟一定程度嘅通脹。


bing       - chinese (simplified)     : 0.2365 BLEU
bing       - chinese (traditional)    : 0.2365 BLEU
bing       - cantonese                : 0.3567 BLEU
google     - chinese                  : 0.2365 BLEU


For that reason we instead switch to the model-based evaluation metric BLEURT

In [104]:
from bleurt import score

scorer = score.BleurtScorer("bleurt-base-128")

INFO:tensorflow:Reading checkpoint bleurt-base-128.


INFO:tensorflow:Reading checkpoint bleurt-base-128.


INFO:tensorflow:Config file found, reading.


INFO:tensorflow:Config file found, reading.


INFO:tensorflow:Will load checkpoint bert_custom


...

INFO:tensorflow:Creating Eager Mode predictor.


INFO:tensorflow:Loading model.


INFO:tensorflow:Loading model.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [164]:
def bleurt(ref, cand):
    return scorer.score(references=[ref], candidates=[cand])[0]

benchmark(all_trans, bleurt, "bleurt")
print_benchmark(all_trans, "bleurt")

100%|██████████| 40/40 [00:06<00:00,  6.36it/s]

bing       - chinese (simplified)     : 0.2806 BLEU
bing       - chinese (traditional)    : 0.2881 BLEU
bing       - cantonese                : 0.3658 BLEU
google     - chinese                  : 0.3008 BLEU


Now that we have a pretty good evaluation measure we can use it to extract sentences that all models do poorly on to see where improvement is needed 

In [168]:
all_trans[0]["bing"]["cantonese"]

{'translation': "He allows me to work according to my own feelings and adjust my teaching style to suit the audience's learning methods.",
 'bleu': 0.3249275698644431,
 'bleurt': 0.17623667418956757}

In [176]:
all_trans[0]["bing"]["cantonese"]

{'translation': "He allows me to work according to my own feelings and adjust my teaching style to suit the audience's learning methods.",
 'bleu': 0.3249275698644431,
 'bleurt': 0.17623667418956757}

In [181]:
filter_value = 0
bad_translations = [trans for trans in all_trans if np.all([np.all([trans[service][lang]["bleurt"] < filter_value for lang in langs[service]]) for service in services])]

In [188]:
print_sentences(bad_translations[0])

original                      : The unrelenting soul-force of those who would hold us accountable blows that all away.
bing   chinese (simplified)   : Those indomitable spiritual forces make us take responsibility and overcome all inner demons.
bing   chinese (traditional)  : Those indomitable spirits compel us to take responsibility and overcome all inner demons.
bing   cantonese              : The indomitable strength of those souls inspires us to take responsibility and overcome all inner demons.
google chinese                : Those indomitable spiritual forces make us take responsibility and overcome all inner demons.
original yue                  : 嗰啲不屈不撓嘅靈魂力量，令我哋負起責任，戰勝所有心魔。
